# 🤖 Working with LLMs in LangChain V1

## Learning Objectives
In this notebook, you will learn:
1. **Provider-Agnostic Initialization** - Use `init_chat_model` to create chat models by name without importing provider-specific classes
2. **Model Comparison** - Query multiple models with the same prompt to compare their responses side by side
3. **Message-Based Conversations** - Use `SystemMessage`/`HumanMessage` objects for fine-grained control over multi-turn conversations
4. **Provider Configuration** - Configure temperature, streaming, and retry behavior for a chat model

## Prerequisites
- Completion of `01_core_concepts.ipynb`
- `OPENAI_API_KEY` set in your `.env` file
- (Optional) `ANTHROPIC_API_KEY` in your `.env` file to run the cross-provider comparison cells

---
## 📦 Setup: Environment & Imports

We load environment variables from `.env` and import the LangChain building blocks used throughout this notebook: `init_chat_model` for provider-agnostic model creation, `ChatOpenAI` for direct OpenAI access, and the message classes used for structured, multi-turn conversations.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Imports and Environment Variables
# ============================================================================
import os

from dotenv import load_dotenv

from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

load_dotenv()

print("✅ Environment variables loaded and LangChain imports ready!")

---
## 🔧 Part 1: Provider-Agnostic Model Initialization

LangChain's `init_chat_model` lets you initialize a chat model by name without importing a provider-specific class — swapping providers (OpenAI, Anthropic, etc.) is just a matter of changing the `model` and `model_provider` arguments. This demo initializes an OpenAI model, invokes it, and shows how the same code would target Anthropic's Claude if an API key is available.

### Key Concepts:
- **`init_chat_model`**: A single factory function that returns the right chat model class based on `model` / `model_provider`
- **Streaming & Retries**: Configured once at initialization (`streaming=True`, `max_retries=3`) and applied to every subsequent call

In [ ]:
# ============================================================================
# MODEL INITIALIZATION: init_chat_model Demo
# ============================================================================
def demo_init_chat_model():
    chat_model = init_chat_model(
        model="gpt-4o-mini",
        # model_provider="openai",
        temperature=0.7,
        streaming=True,
        max_retries=3,
    )

    response = chat_model.invoke("What is the capital of France? Answer in one word.")
    print(f"Response: {response.content}")

    # easy to switch model providers
    # if os.getenv("ANTHROPIC_API_KEY"):
    #     claude = init_chat_model(
    #         model="claude-sonnet-4-5-20250929",
    #         model_provider="anthropic",
    #         temperature=0.7,
    #         streaming=True,
    #         max_retries=3,
    #     )

        # response = claude.invoke("What is the capital of France? Answer in one word.")
        # print(f"Response from Anthropic: {response.content}")

In [ ]:
# ============================================================================
# MODEL INITIALIZATION: Run the Demo
# ============================================================================
demo_init_chat_model()

---
## ⚖️ Part 2: Comparing Multiple Models

Because `init_chat_model` abstracts away provider differences, it's easy to fan the same prompt out to several models and compare their responses side by side. This is useful for quickly evaluating which model gives the best quality, cost, or latency tradeoff for a given task.

In [ ]:
# ============================================================================
# MODEL COMPARISON: Define Comparison Function
# ============================================================================
def demo_model_comparison():
    prompt = "Explain recursion in one sentence."

    models = {
        "gpt-4o-mini": init_chat_model(
            model="gpt-4o-mini",
            temperature=0.7,
            streaming=False,
        ),
        "gpt-4o": init_chat_model(
            model="gpt-4o",
            temperature=0.7,
            streaming=False,
        ),
    }

    # add anthropic model if available
    if os.getenv("ANTHROPIC_API_KEY"):
        models["claude-sonnet-4-5-20250929"] = init_chat_model(
            model="claude-sonnet-4-5-20250929",
            model_provider="anthropic",
            temperature=0.7,
            streaming=False,
        )

    print(f"Prompt: {prompt}\n")

    for model_name, model in models.items():
        response = model.invoke(prompt)
        print(f"Response from {model_name}: {response.content}\n")

In [ ]:
# ============================================================================
# MODEL COMPARISON: Run the Comparison
# ============================================================================
demo_model_comparison()

---
## 💬 Part 3: Message-Based Conversations

Instead of passing a plain string, LangChain lets you pass a list of message objects (`SystemMessage`, `HumanMessage`, `AIMessage`) for precise control over each turn's role. This demo sets a system persona, asks a question, and then continues a multi-turn conversation by appending the model's previous response back onto the message history.

> **Note**: To keep a multi-turn conversation coherent, always append the model's response back onto the `messages` list before sending the next `HumanMessage`.

In [ ]:
# ============================================================================
# MESSAGE-BASED CONVERSATIONS: Define Demo Function
# ============================================================================
def demo_message():
    model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

    # using message objects (more control over roles)
    messages = [
        SystemMessage(content="You are a pirate. Always answer like a pirate."),
        HumanMessage(content="What's the weather like today?"),
    ]
    # print("Using message objects:")
    # print(f"Messages: {messages[0]} | {messages[1]}")

    response = model.invoke(messages)
    print(f"Response from the Pirate: {response.content}")

    # Multi-turn conversation using message objects
    messages.append(response)  # add model's response to the conversation
    messages.append(HumanMessage(content="What about tomorrow?"))

    print("\nMulti-turn conversation:")
    response = model.invoke(messages)
    print(f"Follow-up response from the Pirate: {response.content}")

In [ ]:
# ============================================================================
# MESSAGE-BASED CONVERSATIONS: Run the Demo
# ============================================================================
demo_message()

---
## 🏋️ Part 4: Exercise — Multi-Model Query Function

Now it's your turn. This exercise combines the previous two concepts: build a function that queries several models with the same question and collects each response into a dictionary keyed by model name.

EXERCISE: Create a function that:
1. Takes a question and a list of model names
2. Gets responses from all models
3. Returns a dict of `{model_name: response}`

Test with: `question="What is AI?"`, `models=["gpt-4o-mini", "gpt-4o"]`

In [ ]:
# ============================================================================
# EXERCISE: Multi-Model Query Function
# ============================================================================
def exercise_multi_model():
    """
    EXERCISE: Create a function that:
    1. Takes a question and a list of model names
    2. Gets responses from all models
    3. Returns a dict of {model_name: response}

    Test with: question="What is AI?", models=["gpt-4o-mini", "gpt-4o"]
    """

    def get_responses(question: str, model_names: list[str]) -> dict[str, str]:
        responses = {}
        for model_name in model_names:
            model = init_chat_model(
                model=model_name,
                temperature=0.7,
                streaming=False,
            )
            response = model.invoke(question)
            responses[model_name] = response.content
        return responses

    # Test the function
    results = get_responses("What is AI?", ["gpt-4o-mini", "gpt-4o"])
    for model, answer in results.items():
        print(f"Response from {model}: {answer}\n")

In [ ]:
# ============================================================================
# EXERCISE: Run the Exercise
# ============================================================================
exercise_multi_model()

---
## 📝 Summary

In this notebook, we learned:

### 1. Provider-Agnostic Initialization
- **`init_chat_model()`**: Initialize any supported chat model by name, without importing provider-specific classes
- Switching providers (e.g., OpenAI → Anthropic) only requires changing the `model` and `model_provider` arguments
- Streaming and retry behavior are configured once, at initialization time

### 2. Model Comparison & Message-Based Conversations
- Query multiple models with the same prompt to compare their responses side by side
- Use `SystemMessage` / `HumanMessage` objects for structured, multi-turn conversations
- Append the model's response back onto the message list to maintain conversation history across turns

### Functions Defined
- `demo_init_chat_model()`
- `demo_model_comparison()`
- `demo_message()`
- `exercise_multi_model()`

### Next Steps
- Continue to `03_prompt_messages.ipynb` to explore prompt templates and message formatting in more depth